# Model B, B1 — gaps to the human targets (Oct 30, 2026) — EXPLORATORY, descriptive

**Spec (as given):**

> Descriptive gap table from p24_checks.csv (no simulation): for every statistic and cell, the human value and the gap (sim - human)/CI half-width for L1, final and B1, plus the change B1 - L1 in half-widths. Save it as results/tables/pB1_gaps.csv and print it.
> Also, from the B1 replicates if available (else skip): the median crossing time in smooth vs. stop-and-go crossings, B1 vs. humans, per interface.

**Choices.**
- Every row of `p24_checks.csv` (the 53 check statistics plus the L1b-only rows; `in_check_list` says which).
- Half-width = (ci_hi - ci_lo) / 2 of the human CI used by the checks: the bootstrap CI, or the rule that replaced a zero-width CI (`ci_rule`).
- gap = (sim mean - human) / half-width. A gap in [-1, 1] is about a pass; CIs are not symmetric, so the verdicts in `p24_checks.csv` decide pass / miss, not this table.
- `B1_minus_L1` = gap B1 - gap L1. It is signed: whether B1 is closer to the human value is |gap B1| < |gap L1|.
- **Crossing times, smooth vs. stop-and-go: skipped.** `p24_reps_B1.parquet` holds only the per-replicate check statistics (one median crossing time per cell), not crossings, so the split needs a new simulation.

In [1]:
import numpy as np
import pandas as pd

from slider import config

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)

checks = pd.read_csv(config.P24_CHECKS)
key = ["target", "stat", "topology", "dataset"]
human = checks[checks["preset"] == "L1"].set_index(key)[["in_check_list", "human", "ci_lo", "ci_hi", "ci_rule"]]
human["half_width"] = (human["ci_hi"] - human["ci_lo"]) / 2
gaps = human.drop(columns=["ci_lo", "ci_hi"])
for p in ["L1", "final", "B1"]:
    sim = checks[checks["preset"] == p].set_index(key)
    assert np.allclose(sim["human"], human["human"])                 # the same human targets for every preset
    gaps[f"gap_{p}"] = (sim["sim_mean"] - human["human"]) / human["half_width"]
    gaps[f"verdict_{p}"] = sim["verdict"]
gaps["B1_minus_L1"] = gaps["gap_B1"] - gaps["gap_L1"]
gaps = gaps.reset_index()
gaps.to_csv(config.PB1_GAPS, index=False)
print(gaps.drop(columns=["ci_rule"]).round(2).to_string())

    target                            stat   topology dataset  in_check_list   human  half_width  gap_L1 verdict_L1  gap_final verdict_final  gap_B1 verdict_B1  B1_minus_L1
0        1                success_rate_pct       Full      CC           True   97.22        4.55   -8.74       miss     -11.10          miss   -8.79       miss        -0.06
1        1                success_rate_pct       Full      CL           True   82.54       11.11   -3.80       miss      -4.40          miss   -3.96       miss        -0.16
2        1                success_rate_pct  Hierarchy      CC           True   92.50        7.89   -1.15       miss      -1.78          miss   -1.17       miss        -0.03
3        1                success_rate_pct  Hierarchy      CL           True   95.08        5.26   -6.08       miss      -6.86          miss   -6.18       miss        -0.09
4        1                success_rate_pct  Hierarchy      EL           True   94.12        5.88   -1.25       miss      -2.16         

In [2]:
c = gaps[gaps["in_check_list"]]
closer = (c["gap_B1"].abs() < c["gap_L1"].abs())
print(f"53 check statistics: B1 closer than L1 in {closer.sum()}, farther in {(~closer).sum()}; "
      f"median |gap| L1 {c['gap_L1'].abs().median():.2f}, final {c['gap_final'].abs().median():.2f}, "
      f"B1 {c['gap_B1'].abs().median():.2f}; passes L1 {(c['verdict_L1'] == 'pass').sum()}, "
      f"final {(c['verdict_final'] == 'pass').sum()}, B1 {(c['verdict_B1'] == 'pass').sum()}")
c.assign(abs_change=c["gap_B1"].abs() - c["gap_L1"].abs()).groupby(["target", "stat"]).agg(
    cells=("abs_change", "size"), B1_closer=("abs_change", lambda x: (x < 0).sum()),
    median_B1_minus_L1=("B1_minus_L1", "median")).round(2)

53 check statistics: B1 closer than L1 in 34, farther in 19; median |gap| L1 1.80, final 1.86, B1 1.76; passes L1 17, final 16, B1 17


,,cells,B1_closer,median_B1_minus_L1
target,stat,,,
1,success_rate_pct,6,2,-0.04
2,consensus_time_km_median_s,6,2,0.07
3,fraught_pct_mean,3,2,0.00
4,committed_pct_mean,6,2,0.01
5,breaker_excess,2,2,0.04
6,end_sitting_pct,6,2,-0.00
7,relapse_pct,6,6,-0.13
8,reversal_rate,6,6,-0.09
9,f_rev_median,6,5,0.03


**How to read it.** Each row is one check statistic in one cell. `human` is the target; `gap_*` is how far each preset's mean over 200 replicates sits from it, in human-CI half-widths (positive = the model is above the humans). `B1_minus_L1` is B1's gap minus L1's. The summary counts the check statistics where B1 is closer to the humans than L1. Descriptive only: B1 is EXPLORATORY and decides nothing.

# Model C, C2 — gaps to the human targets (Oct 31, 2026) — EXPLORATORY, descriptive

**Spec (as given):**

> TASK: C2 gap table (descriptive only; no simulation, no model changes). Never load Ring/CT.
>
> 1. From results/tables/p24_checks.csv, extend pB1_gaps.csv with C2:
>    - gap_C2 = (sim - human) / CI half-width;
>    - direction (high / low);
>    - change vs C1 (C2 - C1, in |gap|).
>    Print, for all 53 check statistics plus 10c, one table with these columns:
>      target, stat, cell, human value [CI], sim value and gap for L1, C1, C2,
>      and pass/miss for C2.
>    Sort by target, then cell.
> 2. Summary lines:
>    - median |gap| for L1, final, C1, C2;
>    - number of passes for each;
>    - for C2's misses: how many are high vs low, and how many have |gap| <= 2 (near
>      misses) vs > 2.
> 3. Spell out the Full cells for targets 1, 2, 4, 6, 7: human value, C1, C2. Is C2 still
>    too slow or unsuccessful there, or now overshooting?
> 4. Also commit the C2 R = 200 outputs (p24_reps_C2.parquet, its log, p24_checks.csv,
>    p24_report.txt) if they aren't committed yet.
>
> Notebook: add these cells to pB1_gaps.ipynb, re-execute it, commit, and paste the printed
> tables in your report. Then stop.

**Choices.**
- `p24_checks.csv` holds the C2 R = 200 run (seed 8001). Same half-widths and gap as above.
- The cell above rewrites `pB1_gaps.csv` with L1, final and B1 only, so C1 is added here again, exactly as in `pC_checks.ipynb` (`gap_C1`, `direction_C1`, `verdict_C1`, `C1_minus_L1`), then C2.
- `C2_minus_C1` = |gap C2| - |gap C1| (negative = C2 closer to the humans). Unlike `B1_minus_L1` and `C1_minus_L1`, it is not signed, as the spec asks.
- "53 check statistics plus 10c" = the rows with `in_check_list` plus `pause_share_pct` (10c, 6 cells); the 6 L1b-only between-group SD rows of target 8 are left out of the printed table (they stay in the CSV).
- Pass / miss is the `p24_checks.csv` verdict (10c has one too, though it is not a check statistic).
- Summary lines (part 2) are over the 53 check statistics only. Near miss = a miss with |gap| <= 2.


In [3]:
key = ["target", "stat", "topology", "dataset"]
g = gaps.set_index(key)
for p in ["C1", "C2"]:
    sim = checks[checks["preset"] == p].set_index(key).loc[g.index]
    assert np.allclose(sim["human"], g["human"])                         # the same human targets
    g[f"sim_{p}"] = sim["sim_mean"]
    g[f"gap_{p}"] = (sim["sim_mean"] - g["human"]) / g["half_width"]
    g[f"direction_{p}"] = np.where(g[f"gap_{p}"] > 0, "high", "low")
    g[f"verdict_{p}"] = sim["verdict"]
g["C1_minus_L1"] = g["gap_C1"] - g["gap_L1"]
g["C2_minus_C1"] = g["gap_C2"].abs() - g["gap_C1"].abs()
gaps = g.reset_index()
gaps.drop(columns=["sim_C1", "sim_C2"]).to_csv(config.PB1_GAPS, index=False)

sim_L1 = checks[checks["preset"] == "L1"].set_index(key)["sim_mean"]
ci = checks[checks["preset"] == "L1"].set_index(key)[["ci_lo", "ci_hi"]]
t = gaps[gaps["in_check_list"] | (gaps["stat"] == "pause_share_pct")].copy()
t["cell"] = t["topology"] + " " + t["dataset"]
t["sim_L1"] = sim_L1.loc[t.set_index(key).index].values
t[["ci_lo", "ci_hi"]] = ci.loc[t.set_index(key).index].values

def fmt(x, small):
    return f"{x:.3f}" if small else f"{x:.2f}"

rows = []
for _, r in t.iterrows():
    small = abs(r["human"]) < 1                                          # targets 5 and 9: 3 decimals
    rows.append({"target": r["target"], "stat": r["stat"], "cell": r["cell"],
                 "human [CI]": f"{fmt(r['human'], small)} [{fmt(r['ci_lo'], small)}, {fmt(r['ci_hi'], small)}]",
                 "L1": fmt(r["sim_L1"], small), "gap_L1": f"{r['gap_L1']:.2f}",
                 "C1": fmt(r["sim_C1"], small), "gap_C1": f"{r['gap_C1']:.2f}",
                 "C2": fmt(r["sim_C2"], small), "gap_C2": f"{r['gap_C2']:.2f}",
                 "C2_verdict": r["verdict_C2"]})
table = pd.DataFrame(rows).sort_values(["target", "cell"], kind="stable").reset_index(drop=True)
print(table.to_string())


    target                        stat          cell              human [CI]      L1 gap_L1      C1 gap_C1      C2 gap_C2 C2_verdict
0        1            success_rate_pct       Full CC   97.22 [90.90, 100.00]   57.49  -8.74   60.25  -8.13   81.72  -3.41       miss
1        1            success_rate_pct       Full CL    82.54 [69.84, 92.06]   40.31  -3.80   37.05  -4.09   51.82  -2.76       miss
2        1            success_rate_pct  Hierarchy CC   92.50 [84.21, 100.00]   83.44  -1.15   86.39  -0.77   93.08   0.07       pass
3        1            success_rate_pct  Hierarchy CL   95.08 [89.47, 100.00]   63.06  -6.08   66.37  -5.46   72.50  -4.29       miss
4        1            success_rate_pct  Hierarchy EL   94.12 [88.24, 100.00]   86.75  -1.25   87.67  -1.10   97.05   0.50       pass
5        1            success_rate_pct       Star CC  100.00 [90.97, 100.00]   98.38  -0.36   98.71  -0.29   99.95  -0.01       pass
6        2  consensus_time_km_median_s       Full CC    45.99 [32.72,

**How to read it.** One row per statistic and cell: 53 check statistics plus 10c (`pause_share_pct`), sorted by target, then cell. `human [CI]` is the target and its 95% CI (or the zero-width rule's CI); `L1`, `C1`, `C2` are each model's mean over R = 200 replicates, and `gap_*` is (model - human) / CI half-width, positive = the model is above the humans. `C2_verdict` is the pass / miss from `p24_checks.csv`.

In [4]:
c = gaps[gaps["in_check_list"]]
for p in ["L1", "final", "C1", "C2"]:
    print(f"{p:6s} median |gap| {c[f'gap_{p}'].abs().median():.2f}, passes {(c[f'verdict_{p}'] == 'pass').sum()} of {len(c)}")
m = c[c["verdict_C2"] == "miss"]
print(f"C2 misses: {len(m)}; high {(m['direction_C2'] == 'high').sum()}, low {(m['direction_C2'] == 'low').sum()}; "
      f"|gap| <= 2 (near) {(m['gap_C2'].abs() <= 2).sum()}, > 2 {(m['gap_C2'].abs() > 2).sum()}")
closer = (c["C2_minus_C1"] < 0).sum()
print(f"C2 closer to the humans than C1 in {closer} of {len(c)}, farther in {len(c) - closer}")


L1     median |gap| 1.80, passes 17 of 53
final  median |gap| 1.86, passes 16 of 53
C1     median |gap| 1.20, passes 20 of 53
C2     median |gap| 0.74, passes 31 of 53
C2 misses: 22; high 11, low 11; |gap| <= 2 (near) 9, > 2 13
C2 closer to the humans than C1 in 35 of 53, farther in 18


**How to read it.** Medians and passes are over the 53 check statistics. The C2 miss split uses the sign of the gap (high = above the humans) and its size (near miss = within 2 half-widths). "Closer" compares |gap|: `C2_minus_C1` < 0.

In [5]:
full = gaps[(gaps["topology"] == "Full") & gaps["target"].isin([1, 2, 4, 6, 7])]
full[key[:2] + ["dataset", "human", "half_width", "sim_C1", "gap_C1", "sim_C2", "gap_C2", "verdict_C2",
                "C2_minus_C1"]].round(2).reset_index(drop=True)


,target,stat,dataset,human,half_width,sim_C1,gap_C1,sim_C2,gap_C2,verdict_C2,C2_minus_C1
0,1,success_rate_pct,CC,97.22,4.55,60.25,-8.13,81.72,-3.41,miss,-4.72
1,1,success_rate_pct,CL,82.54,11.11,37.05,-4.09,51.82,-2.76,miss,-1.33
2,2,consensus_time_km_median_s,CC,45.99,13.62,137.58,6.73,84.07,2.80,miss,-3.93
3,2,consensus_time_km_median_s,CL,89.19,23.80,299.90,8.85,269.50,7.57,miss,-1.28
4,4,committed_pct_mean,CC,14.61,4.50,20.63,1.34,29.50,3.31,miss,1.97
5,4,committed_pct_mean,CL,6.50,2.35,5.76,-0.31,7.26,0.32,pass,0.01
6,6,end_sitting_pct,CC,72.82,3.42,76.05,0.94,80.50,2.24,miss,1.30
7,6,end_sitting_pct,CL,60.24,3.67,56.01,-1.15,54.86,-1.46,miss,0.31
8,7,relapse_pct,CC,76.47,8.03,93.20,2.08,90.03,1.69,miss,-0.40
9,7,relapse_pct,CL,67.33,9.80,87.38,2.05,88.93,2.20,miss,0.16


**How to read it.** The Full cells (CC, CL) of targets 1, 2, 4, 6 and 7: the human value, each model's mean and gap, and `C2_minus_C1` (< 0 = C2 closer). Units: 1, 4, 6, 7 in %, 2 in seconds. Answer below.

**Full cells: still too slow and unsuccessful, except two CC overshoots.**
- **Success (1).** CC 81.7% vs human 97.2% (C1 60.3%); CL 51.8% vs 82.5% (C1 37.1%). Both still too low, about a third to a half of C1's gap closed. No overshoot.
- **Consensus time (2).** CC 84 s vs 46 s (C1 138 s); CL 270 s vs 89 s (C1 300 s). Both still too slow; CC is much closer, CL barely moved.
- **Committed (4).** CC 29.5% vs 14.6% (C1 20.6%): C2 now overshoots, farther off than C1. CL 7.3% vs 6.5%: a pass.
- **End sitting (6).** CC 80.5% vs 72.8% (C1 76.1%): C2 overshoots, a miss (C1 passed). CL 54.9% vs 60.2% (C1 56.0%): still too low, a bit farther off.
- **Relapse (7).** CC 90.0% vs 76.5% (C1 93.2%); CL 88.9% vs 67.3% (C1 87.4%). Both still too high, about as far off as C1.

So on Full, C2 groups still agree too rarely and too slowly, most of all on CL. In CC they now spend too much time committed and sitting at the end walls. Descriptive only: C2 is EXPLORATORY and decides nothing.
